# DSL examples dev

Anton Antonov  
May 2026

---

## Setup

In [ ]:
use DSL::Examples;

use Data::TypeSystem;
use Data::Importers;

use Hash::Merge;
use JSON::Fast;

----

## Review and extract English commands

In [ ]:
dsl-examples(from => 'English').map({ $_.values».elems })

In [ ]:
my @commands = dsl-examples(from => 'English').values.map({ $_».keys })».values.flat(:hammer).unique;
@commands.elems

In [ ]:
dsl-examples().keys.map({ $_ => dsl-examples($_, Whatever, 'English')».elems.values.sum })

In [ ]:
my @commands = dsl-examples().keys.map({ dsl-examples($_, Whatever, 'English').map(*.values».keys) }).flat(:hammer).unique;
@commands.elems

In [ ]:
#% html
dsl-examples('Python', Whatever, 'English')
==> to-html

----

## Translation

In [ ]:
my @commands = dsl-examples().keys.map({ dsl-examples($_, Whatever, 'English').map(*.values».keys) }).flat(:hammer).unique;
@commands.elems

In [ ]:
my $conf = llm-configuration('ChatGPT', model => 'gpt-5.3-chat-latest', :8192max-tokens);

In [ ]:
my $res = 
    llm-synthesize([
        'Translate the following commands from English to Brazilian Portuguese",
        "and give the corresponding JSON dictionary:",
        "the keys are English commands the values are corresponding Brazilian Portuguese commands.',
        "Please translate all English commands and do not change them.",
        to-json(@commands),
        llm-prompt('NothingElse')('JSON')
    ],
    e => $conf,
    form => sub-parser('JSON'):drop
);

deduce-type($res)

Make sure all examples in LLM result are found in the original list:

In [ ]:
(@commands (-) $res.keys).elems

### Export 

In [ ]:
#spurt($*CWD ~ '/../resources/english-bulgarian.json', to-json($res, :pretty))
#spurt($*CWD ~ '/../resources/english-russian.json', to-json($res, :pretty))
spurt($*CWD ~ '/../resources/english-portuguese.json', to-json($res, :pretty))

---

## Apply translations

Get the dictionary (derived above):

In [ ]:
#my %dictionary = data-import($*CWD ~ '/../resources/english-bulgarian.json');
#my %dictionary = data-import($*CWD ~ '/../resources/english-russian.json');
my %dictionary = data-import($*CWD ~ '/../resources/english-portuguese.json');
deduce-type(%dictionary)

Get English DSL examples:

In [ ]:
my %english-examples = dsl-examples(from => 'English');
deduce-type(%english-examples)

- For each programming language
  - For each workflow name
    - Replace the key phrase


In [ ]:
my %new-examples;

for %english-examples.kv -> $lang, %workflows {
    my %wt;
    for %workflows.kv -> $workflow, %examples {
        %wt{$workflow} = %examples.map({ %dictionary{$_.key} => $_.value }).Hash    
    }
    %new-examples{$lang} = %wt 
};

deduce-type(%new-examples)

### Export

In [ ]:
spurt($*CWD ~ '/../resources/dsl-examples-portuguese.json', to-json(%new-examples, :pretty))
#spurt($*CWD ~ '/../resources/dsl-examples-russian.json', to-json(%new-examples, :pretty))
#spurt($*CWD ~ '/../resources/dsl-examples-bulgarian.json', to-json(%new-examples, :pretty))

----

## Code examples translation

In [ ]:
#% chat ru prompt
#NothingElse|Raku #Translate|'Brazilian Portuguese'

In [ ]:
#% chat ru
```raku
        English => [
                "Use the time series dfTemperatureData",
                "Echo data summary",
                "Do quantile regression with 20 knots and probabilities: 0.03, 0.5, 0.97",
                "Use \{orange, blue, orange\} for regression curves plotting",
                "Use gray for data plotting",
                "Date plot with aspect ratio 1/3 and image size 1000",
                "Give the error plot with absolute errors",
                "Use light gray, red, and red for data plotting",
                "Use Black for regression curves plotting",
                "Give a date plot for the outliers with aspect ratio 1/3 and image size 1000"
        ]
```
